<a href="https://colab.research.google.com/github/KiranDhanvate/ai-job-aggregator/blob/kiran/convdeepfm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
================================================================================
FILE 2: INFERENCE SCRIPT - integrate_inference.py
Run this EVERY TIME to get job recommendations for a user
================================================================================
"""

# ============================================================================
# CELL 1: Install Dependencies
# ============================================================================
import sys
import subprocess

def install_packages():
    print("=" * 80)
    print("🚀 INSTALLING DEPENDENCIES")
    print("=" * 80)

    packages = [
        'numpy==2.0.2', 'pandas==2.1.4', 'torch', 'scikit-learn',
        'tqdm', 'pdfplumber', 'python-docx', 'python-jobspy',
    ]

    for package in packages:
        try:
            subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', package])
            print(f"✓ {package.split('==')[0]}")
        except:
            print(f"⚠ {package.split('==')[0]} (may need restart)")

    print("\n✅ Installation complete!\n")


In [ ]:
# ============================================================================
# CELL 2: Mount Google Drive
# ============================================================================
from google.colab import drive
drive.mount('/content/drive')

print("\n✅ Google Drive mounted!")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

✅ Google Drive mounted!


In [ ]:
# ============================================================================
# CELL 3: IMPORTS
# ============================================================================

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
import pandas as pd
import json
import re
import os
import pickle
from collections import Counter
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from tqdm import tqdm
from datetime import datetime

# Resume parsing
import pdfplumber
import docx

# Job scraping
!pip install python-jobspy
from jobspy import scrape_jobs

# File upload for Colab
from google.colab import files

print("✅ All libraries imported successfully!")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"💻 Device: {device}\n")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.2/61.2 kB 5.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.5/40.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 89.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 796.9/796.9 kB 53.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 MB 17.1 MB/s eta 0:00:00
  Attempting uninstall: regex
    Found existing installation: regex 2025.11.3
    Uninstalling regex-2025.11.3:
      Successfully uninstalled regex-2025.11.3
  Attempting uninstall: NUMPY
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pytensor 2.36.3 requires numpy>=2.0, but

In [ ]:
# ============================================================================
# CELL 4: Skills Database
# ============================================================================
SKILLS_DATABASE = [
    # Programming Languages
    'Python', 'Java', 'JavaScript', 'TypeScript', 'C++', 'C#', 'C', 'PHP',
    'Ruby', 'Swift', 'Kotlin', 'Go', 'Rust', 'Scala', 'R', 'MATLAB',

    # Web Frontend
    'HTML', 'CSS', 'React', 'Angular', 'Vue.js', 'jQuery', 'Bootstrap',
    'Tailwind CSS', 'SASS', 'LESS', 'Redux', 'Next.js',

    # Web Backend
    'Node.js', 'Express.js', 'Django', 'Flask', 'FastAPI', 'Spring Boot',
    'ASP.NET', 'Ruby on Rails', 'Laravel', 'NestJS',

    # Databases
    'MySQL', 'PostgreSQL', 'MongoDB', 'Redis', 'Oracle', 'SQLite',
    'Cassandra', 'DynamoDB', 'Elasticsearch', 'Neo4j', 'MariaDB',

    # Data Science & ML
    'TensorFlow', 'PyTorch', 'Keras', 'Scikit-learn', 'Pandas', 'NumPy',
    'Matplotlib', 'Seaborn', 'OpenCV', 'NLTK', 'SpaCy', 'Transformers',
    'Machine Learning', 'Deep Learning', 'Neural Networks', 'NLP',
    'Computer Vision', 'Data Analysis',

    # Big Data
    'Spark', 'Hadoop', 'Kafka', 'Airflow', 'Databricks', 'Hive',

    # Cloud Platforms
    'AWS', 'Azure', 'GCP', 'Heroku', 'DigitalOcean', 'Firebase',

    # DevOps & Tools
    'Docker', 'Kubernetes', 'Jenkins', 'Git', 'GitHub', 'GitLab',
    'CI/CD', 'Terraform', 'Ansible', 'Linux', 'Unix', 'Nginx',

    # Mobile Development
    'Android', 'iOS', 'React Native', 'Flutter', 'Xamarin', 'Ionic',

    # Other Technologies
    'REST API', 'GraphQL', 'Microservices', 'WebSocket', 'gRPC',
    'Agile', 'Scrum', 'JIRA', 'Tableau', 'Power BI'
]

In [ ]:
# ============================================================================
# CELL 5: Resume Parser
# ============================================================================
class ResumeParser:
    """Parse resume and extract information"""

    def __init__(self):
        self.skills_db = SKILLS_DATABASE

    def extract_text_from_pdf(self, pdf_path):
        text = ""
        with pdfplumber.open(pdf_path) as pdf:
            for page in pdf.pages:
                page_text = page.extract_text()
                if page_text:
                    text += page_text + "\n"
        return text

    def extract_text_from_docx(self, docx_path):
        doc = docx.Document(docx_path)
        return "\n".join([para.text for para in doc.paragraphs])

    def extract_skills(self, text):
        text_lower = ' ' + text.lower() + ' '
        found = set()
        for skill in self.skills_db:
            pattern = r'\b' + re.escape(skill.lower()) + r'\b'
            if re.search(pattern, text_lower):
                found.add(skill)
        return sorted(list(found))

    def extract_experience(self, text):
        patterns = [
            r'(\d+)\+?\s*(?:years?|yrs?)(?:\s+of)?\s+(?:experience|exp)',
            r'(?:experience|exp)(?:\s+of)?\s+(\d+)\+?\s*(?:years?|yrs?)',
        ]
        for pattern in patterns:
            match = re.search(pattern, text.lower())
            if match:
                return int(match.group(1))
        return 0

    def extract_education(self, text):
        text_lower = text.lower()
        if any(kw in text_lower for kw in ['ph.d', 'phd', 'doctorate']):
            return 'PhD'
        elif any(kw in text_lower for kw in ['master', 'm.s', 'm.tech', 'mba', 'mca']):
            return 'Masters'
        return 'Bachelors'

    def parse_resume(self, file_path, file_type='pdf'):
        print(f"\n📄 Parsing resume: {os.path.basename(file_path)}")
        print("=" * 60)

        # Extract text
        if file_type.lower() == 'pdf':
            text = self.extract_text_from_pdf(file_path)
        elif file_type.lower() in ['docx', 'doc']:
            text = self.extract_text_from_docx(file_path)
        else:
            with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read()

        if not text or len(text) < 50:
            raise ValueError("Could not extract meaningful text from resume")

        # Extract features
        skills = self.extract_skills(text)
        experience = self.extract_experience(text)
        education = self.extract_education(text)

        print(f"✅ Successfully parsed!")
        print(f"   Skills: {len(skills)}")
        print(f"   Experience: {experience} years")
        print(f"   Education: {education}")

        if skills:
            print(f"\n🔧 Top Skills: {', '.join(skills[:10])}")
            if len(skills) > 10:
                print(f"       ... and {len(skills) - 10} more")

        return {
            'skills': skills,
            'experience_years': experience,
            'education_level': education,
            'resume_text': text[:1000]
        }


In [ ]:
# ============================================================================
# CELL 6: Job Scraper & Processor
# ============================================================================
class JobProcessor:
    """Scrape and process job listings"""

    def __init__(self):
        self.skills_db = SKILLS_DATABASE

    def scrape_jobs(self, location="India", results_wanted=100, site_names=['indeed', 'linkedin']):
        print(f"\n🔍 SCRAPING JOBS")
        print("=" * 60)
        print(f"   Location: {location}")
        print(f"   Sites: {', '.join(site_names)}")

        try:
            jobs_df = scrape_jobs(
                site_name=site_names,
                search_term="Software Engineer",
                location=location,
                results_wanted=results_wanted,
                hours_old=168,
                country_indeed='india'
            )

            if jobs_df is not None and len(jobs_df) > 0:
                jobs_df = jobs_df.drop_duplicates(subset=['title', 'company'], keep='first')
                print(f"✅ Scraped {len(jobs_df)} unique jobs")
                return jobs_df.to_dict('records')

            return []

        except Exception as e:
            print(f"❌ Scraping failed: {str(e)}")
            return []

    def extract_skills_from_description(self, description):
        if not description:
            return []

        text_lower = ' ' + str(description).lower() + ' '
        found_skills = []

        for skill in self.skills_db:
            pattern = r'\b' + re.escape(skill.lower()) + r'\b'
            if re.search(pattern, text_lower):
                found_skills.append(skill)

        return sorted(list(set(found_skills)))

    def extract_required_experience(self, description):
        if not description:
            return 3  # Default

        text_lower = str(description).lower()

        patterns = [
            r'(\d+)\+?\s*(?:to|\-)\s*(\d+)\s*(?:years?|yrs?)',
            r'(\d+)\+?\s*(?:years?|yrs?)(?:\s+of)?\s+(?:experience|exp)',
        ]

        for pattern in patterns:
            match = re.search(pattern, text_lower)
            if match:
                if len(match.groups()) > 1 and match.group(2):
                    return int((int(match.group(1)) + int(match.group(2))) / 2)
                return int(match.group(1))

        # Check for level indicators
        if any(term in text_lower for term in ['entry level', 'junior', 'fresher', 'graduate']):
            return 0
        if any(term in text_lower for term in ['senior', 'lead', 'principal', 'staff']):
            return 7

        return 3

    def process_jobs(self, jobs_list):
        print(f"\n🔧 PROCESSING {len(jobs_list)} JOBS")
        print("=" * 60)

        processed = []

        for i, job in enumerate(jobs_list):
            try:
                desc = job.get('description', '')
                if not desc:
                    continue

                skills = self.extract_skills_from_description(desc)
                required_exp = self.extract_required_experience(desc)

                processed.append({
                    'job_id': f"job_{i}",
                    'title': job.get('title', 'Unknown Position'),
                    'company': job.get('company', 'Unknown Company'),
                    'location': job.get('location', 'Not specified'),
                    'job_type': job.get('job_type', 'Full-time'),
                    'description': str(desc)[:500],
                    'full_description': str(desc),
                    'skills': skills,
                    'required_experience': required_exp,
                    'job_url': job.get('job_url', '#'),
                    'date_posted': str(job.get('date_posted', 'Recently'))
                })

            except Exception as e:
                continue

        print(f"✅ Processed {len(processed)} jobs successfully")

        # Show stats
        jobs_with_exp = [j for j in processed if j['required_experience'] is not None]
        if jobs_with_exp:
            avg_exp = np.mean([j['required_experience'] for j in jobs_with_exp])
            print(f"📊 Experience: {len(jobs_with_exp)}/{len(processed)} jobs (avg: {avg_exp:.1f} yrs)")

        return processed


In [ ]:
# ============================================================================
# CELL 7: ConvDeepFM Model Architecture
# ============================================================================
class ConvDeepFM(nn.Module):
    """ConvDeepFM Model - Exact architecture from training"""

    def __init__(self, field_dims, deep_dim, embed_dim=32):
        super().__init__()

        self.embedding = nn.Embedding(sum(field_dims), embed_dim)
        self.offsets = torch.tensor(
            np.array((0, *np.cumsum(field_dims)[:-1])),
            dtype=torch.long
        )
        self.embed_dropout = nn.Dropout(0.2)

        self.convs = nn.ModuleList([
            nn.Conv1d(embed_dim, 64, k) for k in [2, 3, 4]
        ])

        self.deep = nn.Sequential(
            nn.Linear(64 * 3 + deep_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.4),
            nn.Linear(128, 1)
        )

        self.fusion = nn.Linear(2, 1)

    def forward(self, x_cat, x_deep):
        x_cat = x_cat + self.offsets.to(x_cat.device)
        emb = self.embed_dropout(self.embedding(x_cat))

        # FM component
        sum_sq = torch.sum(emb, dim=1) ** 2
        sq_sum = torch.sum(emb ** 2, dim=1)
        fm_out = 0.5 * torch.sum(sum_sq - sq_sum, dim=1, keepdim=True)

        # CNN component
        x = emb.permute(0, 2, 1)
        cnn_out = torch.cat([
            torch.max(torch.relu(conv(x)), dim=2)[0]
            for conv in self.convs
        ], dim=1)

        # Deep component
        deep_out = self.deep(torch.cat([cnn_out, x_deep], dim=1))

        # Fusion (raw score)
        return self.fusion(torch.cat([fm_out, deep_out], dim=1)).squeeze(1)


In [ ]:
# ============================================================================
# CELL 8: Job Recommender (LOADS ALL ARTIFACTS)
# ============================================================================
class ConvDeepFMJobRecommender:
    """Job Recommender - Uses trained model and saved artifacts"""

    def __init__(self, model_dir="/content/drive/MyDrive/models/"):
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.model_dir = model_dir

        # File paths
        self.model_path = os.path.join(model_dir, "convdeepfm_best.pth")
        self.meta_path = os.path.join(model_dir, "convdeepfm_meta.pth")
        self.tfidf_path = os.path.join(model_dir, "tfidf_vectorizer.pkl")
        self.encoders_path = os.path.join(model_dir, "label_encoders.pkl")

        # ID mappings for new users/jobs
        self.user_id_map = {}
        self.job_id_map = {}
        self.next_user_id = 0
        self.next_job_id = 0

        # Scoring weights
        self.model_weight = 0.7
        self.skill_weight = 0.3

        # Load everything
        self.load_trained_model()

    def load_trained_model(self):
        """Load model and ALL preprocessing artifacts"""
        print("\n" + "=" * 80)
        print("📥 LOADING TRAINED ConvDeepFM MODEL & ARTIFACTS")
        print("=" * 80)

        # 1. Load metadata
        print(f"\n1️⃣ Loading metadata...")
        self.metadata = torch.load(self.meta_path, map_location=self.device)
        print(f"   ✅ field_dims: {self.metadata['field_dims']}")
        print(f"   ✅ deep_dim: {self.metadata['deep_dim']}")

        # 2. Load model
        print(f"\n2️⃣ Loading model weights...")
        self.model = ConvDeepFM(
            field_dims=self.metadata['field_dims'],
            deep_dim=self.metadata['deep_dim']
        ).to(self.device)

        checkpoint = torch.load(self.model_path, map_location=self.device)
        self.model.load_state_dict(checkpoint)
        self.model.eval()
        print(f"   ✅ Model loaded: {sum(p.numel() for p in self.model.parameters()):,} parameters")

        # 3. Load TF-IDF
        print(f"\n3️⃣ Loading TF-IDF vectorizer...")
        with open(self.tfidf_path, 'rb') as f:
            self.tfidf = pickle.load(f)
        print(f"   ✅ TF-IDF loaded: {len(self.tfidf.vocabulary_)} vocabulary")

        # 4. Load Label Encoders
        print(f"\n4️⃣ Loading label encoders...")
        with open(self.encoders_path, 'rb') as f:
            self.label_encoders = pickle.load(f)
        print(f"   ✅ Encoders loaded: {list(self.label_encoders.keys())}")
        for key, encoder in self.label_encoders.items():
            print(f"      • {key}: {len(encoder.classes_)} classes")

        print("\n" + "=" * 80)
        print("✅ ALL ARTIFACTS LOADED SUCCESSFULLY!")
        print("=" * 80)

    def get_user_id(self, key):
        """Assign consistent user ID"""
        if key not in self.user_id_map:
            self.user_id_map[key] = self.next_user_id
            self.next_user_id += 1
        return self.user_id_map[key]

    def get_job_id(self, key):
        """Assign consistent job ID"""
        if key not in self.job_id_map:
            self.job_id_map[key] = self.next_job_id
            self.next_job_id += 1
        return self.job_id_map[key]

    def safe_encode(self, encoder, value):
        """Safely encode categorical values"""
        try:
            return encoder.transform([value])[0]
        except ValueError:
            # Fallback to first class if value not seen in training
            return 0

    def prepare_features(self, user_profile, job):
        """Prepare features EXACTLY as in training"""

        # Categorical features (5)
        user_key = f"user_{hash(user_profile['resume_text'])}"
        job_key = job['job_id']

        x_cat = np.array([
            self.get_user_id(user_key),
            self.get_job_id(job_key),
            self.safe_encode(self.label_encoders['user_education'], user_profile['education_level']),
            self.safe_encode(self.label_encoders['job_type'], job['job_type']),
            self.safe_encode(self.label_encoders['location'], job['location'])
        ], dtype=np.int64)

        # Deep features (603 = 300 + 300 + 3)
        user_tfidf = self.tfidf.transform([user_profile['resume_text']]).toarray()
        job_tfidf = self.tfidf.transform([job['description']]).toarray()

        numeric = np.array([[
            user_profile['experience_years'],
            job['required_experience'],
            user_profile['experience_years'] - job['required_experience']
        ]], dtype=np.float32)

        x_deep = np.hstack([user_tfidf, job_tfidf, numeric]).astype(np.float32)

        return x_cat, x_deep

    def predict_score(self, user_profile, job):
        """Get DYNAMIC model prediction"""
        x_cat, x_deep = self.prepare_features(user_profile, job)

        x_cat_t = torch.LongTensor(x_cat).unsqueeze(0).to(self.device)
        x_deep_t = torch.FloatTensor(x_deep).to(self.device)

        with torch.no_grad():
            raw_score = self.model(x_cat_t, x_deep_t).item()

        # Apply sigmoid
        return 1 / (1 + np.exp(-raw_score))

    def recommend_from_resume(self, resume_path, resume_type='pdf', location="India",
                            num_jobs=100, top_k=15, min_skill_match=0.2,
                            site_names=['indeed', 'linkedin']):
        """Complete recommendation pipeline"""

        print("\n" + "=" * 80)
        print("🚀 STARTING JOB RECOMMENDATION")
        print("=" * 80)

        # Parse resume
        parser = ResumeParser()
        user_profile = parser.parse_resume(resume_path, resume_type)

        if not user_profile['skills']:
            print("\n❌ ERROR: No skills found in resume!")
            return None

        # Scrape jobs
        processor = JobProcessor()
        raw_jobs = processor.scrape_jobs(location, num_jobs, site_names)

        if not raw_jobs:
            print("\n❌ ERROR: No jobs found!")
            return None

        # Process jobs
        jobs = processor.process_jobs(raw_jobs)

        if not jobs:
            print("\n❌ ERROR: No valid jobs after processing!")
            return None

        # Generate recommendations
        print("\n" + "=" * 80)
        print("🎯 GENERATING RECOMMENDATIONS")
        print("=" * 80)

        recommendations = []
        user_skills_set = set(user_profile['skills'])

        for job in tqdm(jobs, desc="Scoring jobs"):
            job_skills_set = set(job['skills'])

            # Calculate skill match
            if not job_skills_set:
                skill_match = 0.0
            else:
                skill_match = len(user_skills_set & job_skills_set) / len(job_skills_set)

            if skill_match < min_skill_match:
                continue

            # Get DYNAMIC model score
            model_score = self.predict_score(user_profile, job)

            # Calculate final score
            final_score = (
                self.model_weight * model_score +
                self.skill_weight * skill_match
            )

            # Experience bonus/penalty
            if job['required_experience'] is not None:
                exp_diff = abs(user_profile['experience_years'] - job['required_experience'])
                if exp_diff <= 2:
                    final_score *= 1.1  # 10% boost
                elif exp_diff > 5:
                    final_score *= 0.9  # 10% penalty

            final_score = min(final_score, 1.0)  # Cap at 1.0

            matching_skills = list(user_skills_set & job_skills_set)
            missing_skills = list(job_skills_set - user_skills_set)

            recommendations.append({
                'job': job,
                'model_score': model_score,
                'skill_match': skill_match,
                'final_score': final_score,
                'matching_skills': matching_skills,
                'missing_skills': missing_skills
            })

        # Sort and get top-k
        recommendations.sort(key=lambda x: x['final_score'], reverse=True)
        recommendations = recommendations[:top_k]

        # Display results
        self._display_results(user_profile, recommendations, location)

        return {
            'user_profile': user_profile,
            'recommendations': recommendations,
            'total_jobs_analyzed': len(jobs),
            'location': location,
            'timestamp': datetime.now().isoformat()
        }

    def _display_results(self, user, recs, location):
        """Display recommendations"""
        print("\n" + "=" * 80)
        print("🎯 TOP JOB RECOMMENDATIONS")
        print("=" * 80)

        print(f"\n📊 CANDIDATE PROFILE:")
        print(f"   ✓ Skills: {len(user['skills'])}")
        print(f"   ✓ Experience: {user['experience_years']} years")
        print(f"   ✓ Education: {user['education_level']}")
        print(f"   ✓ Location: {location}")

        print(f"\n⚙️  SCORING CONFIGURATION:")
        print(f"   • Model Weight: {self.model_weight:.0%}")
        print(f"   • Skill Weight: {self.skill_weight:.0%}")
        print(f"   • Experience Bonus: ±10%")

        if recs:
            model_scores = [r['model_score'] for r in recs]
            skill_scores = [r['skill_match'] for r in recs]
            final_scores = [r['final_score'] for r in recs]

            print(f"\n📈 SCORE DISTRIBUTION:")
            print(f"   Model Scores:  {np.min(model_scores):.2f} - {np.max(model_scores):.2f} (avg: {np.mean(model_scores):.2f})")
            print(f"   Skill Matches: {np.min(skill_scores):.2f} - {np.max(skill_scores):.2f} (avg: {np.mean(skill_scores):.2f})")
            print(f"   Final Scores:  {np.min(final_scores):.2f} - {np.max(final_scores):.2f} (avg: {np.mean(final_scores):.2f})")

        print(f"\n" + "=" * 80)
        print(f"TOP {len(recs)} JOB RECOMMENDATIONS")
        print("=" * 80)

        for i, rec in enumerate(recs, 1):
            job = rec['job']

            print(f"\n{i}. {job['title']}")
            print(f"🏢 {job['company']}")
            print(f"📍 {job['location']} | {job['job_type']}")

            if job['required_experience'] is not None:
                exp_diff = user['experience_years'] - job['required_experience']
                exp_emoji = "✅" if abs(exp_diff) <= 2 else "⚠️" if abs(exp_diff) <= 5 else "❌"
                print(f"💼 Experience: {job['required_experience']} years required {exp_emoji} (You: {user['experience_years']})")

            print(f"📅 Posted: {job['date_posted']}")
            print(f"\n📊 Scores:")
            print(f"   • Final Score: {rec['final_score']:.1%}")
            print(f"   • Model Score: {rec['model_score']:.1%} (weight: {self.model_weight:.0%})")
            print(f"   • Skill Match: {rec['skill_match']:.1%} (weight: {self.skill_weight:.0%})")

            if rec['matching_skills']:
                print(f"\n✅ Matching Skills ({len(rec['matching_skills'])}): {', '.join(rec['matching_skills'][:5])}")
                if len(rec['matching_skills']) > 5:
                    print(f"       ... and {len(rec['matching_skills']) - 5} more")

            if rec['missing_skills'][:3]:
                print(f"📚 Skills to Learn: {', '.join(rec['missing_skills'][:3])}")

            print(f"\n🔗 Apply: {job['job_url'][:60]}...")
            print("-" * 80)


In [ ]:
# ============================================================================
# CELL 9: MAIN EXECUTION
# ============================================================================
print("\n" + "=" * 80)
print("✅ ConvDeepFM JOB RECOMMENDATION SYSTEM READY!")
print("=" * 80)

# Upload resume
print("\n📤 UPLOAD YOUR RESUME")
print("=" * 80)
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    file_type = filename.split('.')[-1].lower()

    print(f"\n✅ Uploaded: {filename}")

    # Create recommender
    engine = ConvDeepFMJobRecommender(
        model_dir="/content/drive/MyDrive/models/"
    )

    # Get recommendations
    result = engine.recommend_from_resume(
        resume_path=filename,
        resume_type=file_type,
        location='India',
        num_jobs=100,
        top_k=15,
        min_skill_match=0.2,
        site_names=['indeed', 'linkedin']
    )

    # Save results
    if result:
        with open('job_recommendations.json', 'w') as f:
            json.dump(result, f, indent=2, default=str)
        print("\n✅ Results saved to job_recommendations.json")

print("\n" + "=" * 80)
print("🎯 SYSTEM FEATURES:")
print("  ✓ Trained on 11,766 samples")
print("  ✓ Loads SAME TF-IDF & encoders as training")
print("  ✓ FULLY DYNAMIC predictions from trained model")
print("  ✓ Real-time job scraping from Indeed & LinkedIn")
print("=" * 80)


✅ ConvDeepFM JOB RECOMMENDATION SYSTEM READY!

📤 UPLOAD YOUR RESUME


Saving kiran_resumme_java.pdf to kiran_resumme_java.pdf

✅ Uploaded: kiran_resumme_java.pdf

📥 LOADING TRAINED ConvDeepFM MODEL & ARTIFACTS

1️⃣ Loading metadata...
   ✅ field_dims: [1000, 443, 14, 6, 18]
   ✅ deep_dim: 421

2️⃣ Loading model weights...
   ✅ Model loaded: 144,740 parameters

3️⃣ Loading TF-IDF vectorizer...
   ✅ TF-IDF loaded: 209 vocabulary

4️⃣ Loading label encoders...
   ✅ Encoders loaded: ['user_id', 'job_id', 'user_education', 'job_type', 'location']
      • user_id: 1000 classes
      • job_id: 443 classes
      • user_education: 14 classes
      • job_type: 6 classes
      • location: 18 classes

✅ ALL ARTIFACTS LOADED SUCCESSFULLY!

🚀 STARTING JOB RECOMMENDATION

📄 Parsing resume: kiran_resumme_java.pdf
✅ Successfully parsed!
   Skills: 20
   Experience: 0 years
   Education: Bachelors

🔧 Top Skills: Agile, Azure, CI/CD, Docker, FastAPI, Flask, Git, GitHub, Java, Keras
       ... and 10 more

🔍 SCRAPING JOBS
   Location: Pune
   Sites: indeed, linkedin
✅ Scrap

Scoring jobs: 100%|██████████| 154/154 [00:00<00:00, 1250.22it/s]


🎯 TOP JOB RECOMMENDATIONS

📊 CANDIDATE PROFILE:
   ✓ Skills: 20
   ✓ Experience: 0 years
   ✓ Education: Bachelors
   ✓ Location: Pune

⚙️  SCORING CONFIGURATION:
   • Model Weight: 70%
   • Skill Weight: 30%
   • Experience Bonus: ±10%

📈 SCORE DISTRIBUTION:
   Model Scores:  0.71 - 1.00 (avg: 0.92)
   Skill Matches: 0.38 - 0.82 (avg: 0.52)
   Final Scores:  0.76 - 0.92 (avg: 0.83)

TOP 15 JOB RECOMMENDATIONS

1. Senior Software Maintenance Engineer
🏢 Red Hat
📍 MH, IN | fulltime
💼 Experience: 0 years required ✅ (You: 0)
📅 Posted: 2026-01-23

📊 Scores:
   • Final Score: 92.3%
   • Model Score: 98.5% (weight: 70%)
   • Skill Match: 50.0% (weight: 30%)

✅ Matching Skills (7): Git, Python, GitHub, Agile, CI/CD
       ... and 2 more
📚 Skills to Learn: GitLab, Jenkins, Machine Learning

🔗 Apply: https://in.indeed.com/viewjob?jk=8b32613c27605455...
--------------------------------------------------------------------------------

2. Android Developer
🏢 Aidify Learning and Mobility Pvt Ltd
📍 

In [ ]:
# ============================================================================
# CELL 9: MAIN EXECUTION
# ============================================================================
print("\n" + "=" * 80)
print("✅ ConvDeepFM JOB RECOMMENDATION SYSTEM READY!")
print("=" * 80)

# Upload resume
print("\n📤 UPLOAD YOUR RESUME")
print("=" * 80)
uploaded = files.upload()

if uploaded:
    filename = list(uploaded.keys())[0]
    file_type = filename.split('.')[-1].lower()

    print(f"\n✅ Uploaded: {filename}")

    # 🔹 NEW: Take dynamic inputs
    location = input("\n📍 Enter job location (e.g., India, Pune, Remote): ").strip()
    sites_input = input(
        "🌐 Enter job sites (comma-separated: indeed, linkedin): "
    ).strip()

    site_names = [site.strip().lower() for site in sites_input.split(",") if site.strip()]

    # Create recommender
    engine = ConvDeepFMJobRecommender(
        model_dir="/content/drive/MyDrive/models/"
    )

    # Get recommendations
    result = engine.recommend_from_resume(
        resume_path=filename,
        resume_type=file_type,
        location=location,
        num_jobs=100,
        top_k=15,
        min_skill_match=0.2,
        site_names=site_names
    )

    # Save results
    if result:
        with open('job_recommendations.json', 'w') as f:
            json.dump(result, f, indent=2, default=str)
        print("\n✅ Results saved to job_recommendations.json")

print("\n" + "=" * 80)
print("🎯 SYSTEM FEATURES:")
print("  ✓ Trained on 11,766 samples")
print("  ✓ Loads SAME TF-IDF & encoders as training")
print("  ✓ FULLY DYNAMIC predictions from trained model")
print("  ✓ Real-time job scraping from Indeed & LinkedIn")
print("=" * 80)



✅ ConvDeepFM JOB RECOMMENDATION SYSTEM READY!

📤 UPLOAD YOUR RESUME


Saving kiran_resumme1.pdf to kiran_resumme1 (5).pdf

✅ Uploaded: kiran_resumme1 (5).pdf

📍 Enter job location (e.g., India, Pune, Remote): usa
🌐 Enter job sites (comma-separated: indeed, linkedin): indeed, linkedin

📥 LOADING TRAINED ConvDeepFM MODEL & ARTIFACTS

1️⃣ Loading metadata...
   ✅ field_dims: [1000, 443, 14, 6, 18]
   ✅ deep_dim: 421

2️⃣ Loading model weights...
   ✅ Model loaded: 144,740 parameters

3️⃣ Loading TF-IDF vectorizer...
   ✅ TF-IDF loaded: 209 vocabulary

4️⃣ Loading label encoders...
   ✅ Encoders loaded: ['user_id', 'job_id', 'user_education', 'job_type', 'location']
      • user_id: 1000 classes
      • job_id: 443 classes
      • user_education: 14 classes
      • job_type: 6 classes
      • location: 18 classes

✅ ALL ARTIFACTS LOADED SUCCESSFULLY!

🚀 STARTING JOB RECOMMENDATION

📄 Parsing resume: kiran_resumme1 (5).pdf
✅ Successfully parsed!
   Skills: 16
   Experience: 0 years
   Education: Bachelors

🔧 Top Skills: Azure, CI/CD, Docker, Flask, Git, GitHu